# Ejercicio 5. ¿Puede un ESP32 usar un LLM?
**Sí, como cliente de un LLM externo.** El microcontrolador obtiene datos y se comunica con un servidor local que ejecuta Ollama o con un servicio remoto. Ejecutar un LLM general de miles de millones de parámetros directamente en un ESP32 clásico no es viable para esta aplicación.

Espressif documenta para la familia ESP32 clásica hasta 240 MHz y 520 KB de SRAM interna. Esa memoria se comparte con el programa, comunicaciones y buffers; no está toda libre. Otras variantes y placas tienen características distintas y pueden añadir PSRAM (Espressif Systems, s. f.).

| Restricción | Consecuencia | Decisión propuesta |
|---|---|---|
| RAM y flash | No caben pesos ni contexto de un LLM habitual | Inferencia en servidor |
| Cómputo | Generación lenta sin capacidad adecuada | ESP32 adquiere y transmite datos |
| Red y latencia | Puede perder conexión o tardar | Timeout y funcionamiento local determinista |
| Energía | Wi-Fi consume recursos | Muestreo periódico y mensajes cortos |
| Seguridad | Servicios o credenciales expuestos | Gateway autenticado y TLS verificable |
| Salida probabilística | Puede recomendar acciones erróneas | LLM explica; control e interbloqueos locales |

**Aplicación propuesta:** ESP32 registra temperatura de una maqueta y envía lecturas al gateway. El gateway conserva datos, consulta documentos y genera una explicación para el operador. La protección del equipo funciona con lógica determinista independiente del LLM. El umbral del ejemplo es didáctico y no representa un límite industrial validado.

In [1]:
# Estimación didáctica de almacenamiento mínimo de pesos, no memoria total.
parameters=1_000_000_000
bits=4
weight_bytes=parameters*bits/8
sram_bytes=520*1024
print(f'Pesos de 1B a 4 bits: {weight_bytes/1e6:.0f} MB decimales')
print(f'SRAM clásica: {sram_bytes/1024:.0f} KiB nominales')
print(f'Relación pesos/SRAM: {weight_bytes/sram_bytes:.0f} veces')
assert weight_bytes>sram_bytes
# No incluye metadatos de cuantización, activaciones, caché KV ni runtime.

Pesos de 1B a 4 bits: 500 MB decimales
SRAM clásica: 520 KiB nominales
Relación pesos/SRAM: 939 veces


In [2]:
# Simulación ejecutable del comportamiento local: no llama al LLM.
import math

def process_reading(temperature,server_available):
    if type(temperature) not in (int,float) or not math.isfinite(temperature):
        return {'estado':'ERROR_SENSOR','alarma_local':True,'consulta_llm':False}
    alarm=temperature>=60 # Umbral de maqueta; requiere diseño específico en una planta.
    return {'estado':'ALERTA' if alarm else 'NORMAL','alarma_local':alarm,
            'consulta_llm':bool(server_available),'temperatura':temperature}
assert process_reading(70,False)['alarma_local'] is True
assert process_reading(25,False)['consulta_llm'] is False
assert process_reading(float('nan'),True)['estado']=='ERROR_SENSOR'
print('Pruebas aprobadas: alarma local sin servidor y sensor inválido.')

Pruebas aprobadas: alarma local sin servidor y sensor inválido.


### Pseudocódigo del firmware y contrato del gateway
```text
cada intervalo:
    leer sensor y validar rango físico y estado del sensor
    ejecutar protección local diseñada para el equipo
    si hay conexión:
        enviar JSON por HTTPS al gateway con timeout
        validar estado HTTP y tamaño de respuesta
        mostrar explicación al operador
    si falla la conexión:
        conservar control local y registrar evento
```
Entrada propuesta del gateway: `{"equipo":"MAQUETA-01","temperatura_c":42.0,"unidad":"C","secuencia":123}`. El gateway valida esquema, autentica el dispositivo y llama al LLM. Su respuesta debe separar `explicacion` de `fuentes`; no debe transformarse automáticamente en órdenes de actuadores.

Este notebook presenta una arquitectura y simulación, **no un firmware cargado en hardware**. Para implementarlo faltan modelo exacto de placa, sensor, requisitos de seguridad, red, certificados y pruebas de latencia. TinyML para clasificación o detección de anomalías es otra opción local; no equivale a ejecutar un chatbot LLM.

### Referencias
Espressif Systems. (s. f.). *ESP32 series datasheet*. https://documentation.espressif.com/esp32_datasheet_en.html

Ollama. (s. f.). *Generate a chat message*. https://docs.ollama.com/api/chat